<a href="https://colab.research.google.com/github/soule-geophysics/geol-333-714/blob/main/notebooks/HW0_pendulum.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> **Do this first: File > Save a copy in Drive.**
> You are viewing a shared notebook. Anything you type here is NOT saved.
> Save your own copy now, work only in the copy, and rename it
> `HW0_LASTNAME.ipynb` (exact filename and due date in the assignment
> calendar on Brightspace).

# HW0: Pendulum Gravity

**Course:** GEOL 333 / 714, Geophysical Exploration Methods, Fall 2026
**Instructor:** Dax Soule (dax.soule@qc.cuny.edu)
**Due:** Wednesday, September 16, 2026, 11:59 PM
**Submit:** this notebook, renamed `HW0_LASTNAME.ipynb`, to the Brightspace HW0 dropbox
**Estimated time:** about 2 hours 15 minutes across 6 parts (setup, data selection and submission included)

**What you will do**

1. Compute $g$ from a single pendulum measurement, and learn enough Python to do it (variables, f-strings, lists).
2. Load a 25-trial pendulum dataset into Pandas and summarize it.
3. Make two plots: raw period $T$ vs length $l$, and stacked means of $T^2$ vs $l$ with error bars.
4. Estimate the slope of $T^2$ vs $l$ by eye, then fit it with least squares, turn the slope into $g$, and read the fitted intercept as a measurement of the apparatus.
5. Put an error bar on your $g$ and compare it, with its uncertainty, to 9.81 m/s$^2$.
6. Reflect on which hidden assumptions of $g = GM/R^2$ this experiment did, and did not, touch.

**What you will hand in**

This same notebook with all code cells run, every *(Your answer):* filled in. Save it as `HW0_LASTNAME.ipynb` (for example `HW0_Smith.ipynb`) and upload it to the Brightspace HW0 dropbox.

## Your data (~2 min)

Use either dataset.

1. **The Wk 2 in-class CSV.** We collect it together on the classroom apparatus on Sep 9 and it posts that evening at [the Wk 2 in-class CSV on the public course-data mirror](https://raw.githubusercontent.com/soule-geophysics/geol-333-714/main/data/pendulum_inclass.csv). Re-point the `DATA_URL` line in Setup at that URL.
2. **The posted sample CSV**, the Setup default. Synthetic, generated with a fixed random seed to mimic the classroom ringstand apparatus: five lengths, five trials, each `period_s` value a 10-swing time divided by 10.


## Getting unstuck

Stuck? Climb this ladder in order:

1. Re-read this section's markdown, then Runtime > Run all above (cells must run in order).
2. Check the previous self-check cell. If your value is out of range, the problem is upstream of where it surfaced.
3. Re-read Burger §6.2.1 and your notes from the Wk 1 and Wk 2 board work (HW0 has no videos).
4. Post on the **Ask the Class (General Q&A)** discussion topic. The instructor responds within 24 hours, and a classmate may answer sooner.
5. Paste your own code and its error message into CUNY Copilot (sign in at microsoft365.com/chat with your CUNY Login) to understand why it fails.

## Part 0: Getting started in Colab (~10 min)

New to Colab or to Python? Read this once; it is everything you need to run this notebook. Used Colab before? Skim it and jump to Setup.

**What this is.** You are in a Google Colab notebook: a page of *cells* stacked top to bottom, of two kinds. **Text cells** (like this one) hold instructions and questions. **Code cells** (shaded, with a play button on the left) hold Python you run.

**How to run a cell.** Click a code cell, then press **Shift+Enter** (or click the play button). Python runs the cell and prints any output just beneath it. Shift+Enter also drops you onto the next cell, so you can walk down the notebook by pressing it again and again.

**Run in order, top to bottom.** Later cells use values defined earlier (a table, a number). Run them out of order, or skip one, and you get errors like `NameError: name 'df' is not defined`. The reset is **Runtime > Run all**, which runs everything from the top. Use it whenever something looks off.

**When something breaks.** An error prints a red box; its last line names the problem (`NameError`, `ModuleNotFoundError`, `FileNotFoundError`). Read that line first. Most first-week errors clear with **Runtime > Run all** (cells ran out of order) or **Runtime > Restart runtime** then Run all (a stuck session). You cannot harm your computer or the data; the worst case is restarting the runtime.

**Your work lives in your copy.** You already saved a copy to your Drive (the cell at the very top). What you type and run autosaves there; the shared original is read-only.

Now run the cell below to confirm Colab works. It is the simplest possible Python.

In [ ]:
# This is a code cell. Click it, then press Shift+Enter to run it.
# Lines that start with # are comments: notes for humans that Python ignores.
print("Colab is working. I am ready to start HW0.")

# Python is also a calculator. Run this and read the output below the cell.
print("2 + 2 =", 2 + 2)

> **Did it work?** Two lines should appear just below the cell: `Colab is working. I am ready to start HW0.` and `2 + 2 = 4`. If you see them, you have run your first Python and you are ready for Setup. If you got a red error box instead, choose **Runtime > Run all** from the menu and try again.

## Setup (~5 min)

This notebook uses three Python libraries: **NumPy** (numbers and arrays), **Pandas** (tables), and **Plotly** (interactive plots). In Colab all three are already installed. The first cell below checks for them and quietly installs any that are missing, so the notebook runs the same way outside Colab; in Colab it finishes in about a second. The second cell imports them and loads the data.

The data loads straight from a web address (`DATA_URL`), so on the default path there is nothing to download. If the URL is unreachable, or when you switch to the Wk 2 in-class CSV:

1. Download the CSV from the Week 1 Guide on Brightspace.
2. In Colab, click the **Files** icon in the left sidebar.
3. Drag the CSV into the file panel.
4. Change the `DATA_URL` line to the bare filename (for example `DATA_URL = "pendulum_sample.csv"`) and re-run the cell.

If Colab cannot find the file you will get a `FileNotFoundError`. Re-do step 3.

In [ ]:
# Make sure the libraries this notebook needs are available.
# In Colab they are pre-installed, so this is instant. If one is missing
# (a different setup, or a fresh runtime), it installs quietly. Run it once.
import importlib
import importlib.util
import subprocess
import sys

for package in ["numpy", "pandas", "plotly"]:
    if importlib.util.find_spec(package) is None:
        print(f"installing {package} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

print("dependencies ready: numpy, pandas, plotly")

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

# Course accessibility default: colorblind-safe qualitative palette
px.defaults.color_discrete_sequence = px.colors.qualitative.Safe

# Primary data path: stable public URL (no upload needed).
# This points at the SAMPLE CSV. To use the Wk 2 in-class CSV instead,
# change this ONE line; nothing else changes.
DATA_URL = "https://raw.githubusercontent.com/soule-geophysics/geol-333-714/main/data/pendulum_sample.csv"

# Fallback (Brightspace download): download pendulum_sample.csv from the
# Brightspace HW0 page, drag it into Colab's Files panel, then use:
# DATA_URL = "pendulum_sample.csv"

# Instructor local test (repo checkout): uncomment the next line.
# DATA_URL = "../outputs/data/pendulum_sample.csv"

df = pd.read_csv(DATA_URL)
print(f'loaded {len(df)} rows and {len(df.columns)} columns from {DATA_URL}')

> **Colab deletes uploaded files when the runtime disconnects. This is normal.**
> If a CSV you uploaded by hand has vanished, nothing is wrong and you lost no
> work: re-run the data-loading cell above (the URL load restores the data) or
> re-upload the file. Your code and written answers are safe as long as you are
> working in your own saved copy.

## Part 1: One swing, one g (~15 min)

From class (Burger §6.2.1): a pendulum of length $l$ swinging through a small angle has period

$$T = 2\pi\sqrt{\frac{l}{g}}$$

Notice what is *not* in that equation: the mass of the bob, and (so long as it stays small) the amplitude of the swing. Solve for $g$:

$$g = \frac{4\pi^2 l}{T^2}$$

One measured $(l, T)$ pair is enough to compute $g$. In the Wk 1 practice we did this by hand with the hypothetical reading $T = 2.006$ s at $l = 1.000$ m. The cell below redoes that arithmetic in Python, and then repeats it for a stopwatch measurement from the sample CSV's synthetic apparatus: at $l = 1.0$ m, 10 complete swings took 20.18 s.

A trial never times a single swing: the class protocol times **10 complete swings and divides by 10**. Question 1.2 asks you why.

**Question 1.1.** Before you run the cell, predict: the sample-CSV measurement ($T = 2.018$ s) is slightly *longer* than the hypothetical ($T = 2.006$ s) at the same length. Will the computed $g$ land above or below 9.81 m/s$^2$?

*(Your answer):*

In [ ]:
# Variables: a name holds a value. Comments (after #) are notes Python ignores.
L = 1.0               # pendulum length, in meters
t_ten_swings = 20.18  # stopwatch time for 10 complete swings, in seconds

# Arithmetic: / divides, ** raises to a power, np.pi is 3.14159...
T = t_ten_swings / 10                 # period of ONE swing, in seconds
g_one_pair = 4 * np.pi**2 * L / T**2

# f-strings: put f before the quote, and {variables} drop into the text.
# The :.3f means "show 3 digits after the decimal point".
print(f'period of one swing:      T = {T:.3f} s')
print(f'g from this (l, T) pair:  {g_one_pair:.3f} m/s^2')

# The Wk 1 in-class hypothetical, for comparison:
g_hypothetical = 4 * np.pi**2 * 1.000 / 2.006**2
print(f'g from the hypothetical:  {g_hypothetical:.3f} m/s^2')

One more Python idea before the data arrives: a **list** holds several values, and a **for loop** repeats a computation for each one. The cell below predicts the period at three lengths assuming $g = 9.81$ m/s$^2$ (the Wk 1 board practice, now in code). Keep the predictions in mind: Part 2 shows how close the measured data comes.

In [ ]:
# Lists and loops: predict T at three lengths, assuming g = 9.81 m/s^2
g_accepted = 9.81  # m/s^2

for L_pred in [0.4, 0.8, 1.2]:                         # a list of lengths, in meters
    T_pred = 2 * np.pi * np.sqrt(L_pred / g_accepted)  # the pendulum equation
    print(f'l = {L_pred:.1f} m  ->  predicted T = {T_pred:.3f} s')

> **Self-check (Part 1).** `g_one_pair` should land between 9.5 and 9.9 m/s$^2$ (ours: 9.694), and the hypothetical should give 9.811. The three predicted periods should be 1.269, 1.794, and 2.198 s. If `g_one_pair` came out near 0.1, `T` is still the 10-swing total: check the divide-by-10 line. If it came out near 970, `L` is probably in centimeters.

**Question 1.2.** Why does the protocol time 10 complete swings and divide by 10, instead of timing one swing? Estimate it: if your reaction time is about 0.2 s, how large an error does that put on $T$ each way?

*(Your answer):*

**Question 1.3.** Change `g_accepted` to 1.62 (the Moon) and re-run the loop cell. Which way do the predicted periods move, and does that direction make physical sense? Set it back to 9.81 and re-run before moving on.

*(Your answer):*

## Part 2: Load the data into Pandas (~15 min)

Setup already ran the most important line in the notebook:

```python
df = pd.read_csv(DATA_URL)
```

`pd.read_csv` fetched a plain text file of comma-separated values and turned it into a **DataFrame** (a table with named columns) called `df`. The pendulum file has three columns: `trial` (1 to 25), `length_m` (pendulum length in meters), and `period_s` (the period of one swing in seconds; each value is already a 10-swing time divided by 10).

Three commands tell you what is inside any DataFrame.

In [ ]:
# .head() shows the first rows of the table
df.head()

In [ ]:
# .describe() summarizes every numeric column
df.describe()

The real structure here is five stacks of five trials. `groupby('length_m')` splits the table into the five stacks, and `.agg(...)` computes statistics inside each stack. The extra column is the **standard error of the mean**, $\mathrm{SE} = \sigma/\sqrt{N}$, from the Wk 2 stats block: the uncertainty of the *mean* of $N$ trials, not of one trial.

In [ ]:
# Split into one group per length, then compute statistics within each group
stats_T = df.groupby('length_m')['period_s'].agg(['mean', 'std', 'count'])

# Standard error of the mean: std / sqrt(N)
stats_T['se'] = stats_T['std'] / np.sqrt(stats_T['count'])

stats_T.round(4)

> **Self-check (Part 2).** `df.describe()` should report `count = 25` for every column. In `stats_T`, every `count` should be 5, the mean period at 1.0 m should be 2.014 s, and the `std` values should all sit between 0.010 and 0.020 s. If the counts are not 25 and 5, the CSV did not load cleanly: re-run Setup and read its error message.

**Question 2.1.** From `df.describe()`: what are the minimum and maximum periods in the dataset? Using `stats_T`, which length does each belong to?

*(Your answer):*

**Question 2.2.** Read down the `std` column of `stats_T`. The scatter is roughly the same (0.01 to 0.02 s) at every length, even though the period itself nearly doubles from 1.3 s to 2.2 s. What does that pattern suggest about the *source* of the scatter: the pendulum itself, or the hand holding the stopwatch?

*(Your answer):*

## Part 3: Two plots (~25 min)

The model predicts $T = 2\pi\sqrt{l/g}$, so plotting raw $T$ against $l$ should give a curve (a square root), not a line. Squaring both sides gives

$$T^2 = \frac{4\pi^2}{g}\,l$$

which says $T^2$ against $l$ should be a **straight line** with slope $4\pi^2/g$.

That form assumes an ideal pendulum: all the mass at a point, at exactly the length you wrote down. A real apparatus has a bob with size and a string with a knot, so Burger (Eq. 6.6) writes the real period as $T = 2\pi\sqrt{K/g}$, where $K$ is a constant belonging to that particular pendulum. Write $K = l + \varepsilon$, with $\varepsilon$ the fixed offset between the length you record and the length the bob swings on, and squaring gives

$$T^2 = \frac{4\pi^2}{g}\,l \;+\; \frac{4\pi^2}{g}\,\varepsilon$$

So the line has a **slope** and an **intercept**, and each carries something: the slope gives $g$, and the intercept divided by the slope gives $\varepsilon$. Part 4 fits both. One job of this part is to check the straight-line claim against the data; the other is to see what averaging ("stacking") five trials buys us.

### 3a. Raw periods

In [ ]:
# All 25 trials: period T against length L
fig = px.scatter(
    df,
    x='length_m',
    y='period_s',
    title='Raw pendulum periods: T vs l (25 trials)',
    labels={'length_m': 'Pendulum length l (m)', 'period_s': 'Period T (s)'},
)
fig.update_traces(marker=dict(size=10))
fig.show()

**Figure description:** Scatter plot. Horizontal axis: pendulum length in meters, from 0.4 to 1.2. Vertical axis: period of one swing in seconds. There are 25 markers in five vertical clusters, one cluster of five trials at each length. To read this figure without rendering it, use the `linearity_table` printout after the next code cell: it lists the mean of $T$ at each length and the step from each mean to the next.

### 3b. Stacked means of T squared, with error bars

Now the linear version. We add a $T^2$ column, stack the five trials at each length into one mean, and put a $\pm 1$ SE error bar on each mean. The printed table also lists the **step** from each mean to the next. The lengths go up in equal steps of 0.2 m, so a straight line must produce equal steps in the means, while a curve produces steps that grow or shrink. That gives you a way to judge straightness from the numbers as well as from the picture.

In [ ]:
# A new column: period squared, in s^2
df['period_sq'] = df['period_s'] ** 2

# Stack the 5 trials at each length: mean, scatter, and SE of T^2
stats_T2 = df.groupby('length_m')['period_sq'].agg(['mean', 'std', 'count'])
stats_T2['se'] = stats_T2['std'] / np.sqrt(stats_T2['count'])

# Side-by-side table: T and T^2 means, plus the step between consecutive means
linearity_table = pd.DataFrame({
    'T_mean_s': stats_T['mean'],
    'T_step_s': stats_T['mean'].diff(),
    'T2_mean_s2': stats_T2['mean'],
    'T2_step_s2': stats_T2['mean'].diff(),
    'T2_se_s2': stats_T2['se'],
})
linearity_table.round(4)

In [ ]:
# The stacked means of T^2, with +/- 1 SE error bars
means_T2 = stats_T2.reset_index()  # turn the length index back into a column

fig = px.scatter(
    means_T2,
    x='length_m',
    y='mean',
    error_y='se',
    title='Stacked means: T squared vs l, with error bars',
    labels={'length_m': 'Pendulum length l (m)', 'mean': 'Mean of T squared (s^2)'},
)
fig.update_traces(marker=dict(size=12))
fig.show()

**Figure description:** Scatter plot with error bars. Horizontal axis: pendulum length in meters, 0.4 to 1.2. Vertical axis: mean of $T^2$ in s$^2$, one marker per length (five markers). Each marker carries a vertical bar of $\pm 1$ standard error, about 0.02 s$^2$ tall, smaller than the marker symbol at this scale. Every plotted value, and the step between consecutive means, is in the `linearity_table` printout above.

> **Self-check (Part 3).** The five `T2_mean_s2` values should read about 1.626, 2.392, 3.224, 4.057, and 4.880, and every `T2_se_s2` should sit between 0.015 and 0.030. If the `T2` columns look identical to the `T` columns, the squaring line did not run: re-run the cell that creates `period_sq`. If the error bars look enormous, check that `error_y='se'` and not `'std'`.

**Question 3.1.** One of the two plots is a straight line; the other is not. Which is which? Defend the call with numbers from `linearity_table`: across the equal 0.2 m steps in length, compare how the `T_step_s` column behaves with how the `T2_step_s2` column behaves.

*(Your answer):*

**Question 3.2.** At each length the single-trial scatter in $T^2$ (`std`) is about 0.05 s$^2$, but the error bar on the stacked mean (`se`) is about 0.02 s$^2$. What did stacking five trials buy, and what formula is behind it? How many trials per length would it take to shrink the error bars by another factor of 2?

*(Your answer):*

## Part 4: The slope, by eye and by least squares (~20 min)

The fit returns two numbers and both of them mean something:

$$T^2 = \underbrace{\frac{4\pi^2}{g}}_{\text{slope}}\, l \;+\; \underbrace{\frac{4\pi^2}{g}\,\varepsilon}_{\text{intercept}}\qquad\Longrightarrow\qquad g = \frac{4\pi^2}{\text{slope}}, \qquad \varepsilon = \frac{\text{intercept}}{\text{slope}}$$

The **slope** carries the physics: it gives $g$. The **intercept** carries the apparatus: it gives $\varepsilon$, the fixed offset between the length written in the table and the length the bob actually swings on. A constant offset in every recorded length shifts the line up or down without tilting it, which is why $g$ comes off the slope and survives the offset.

First estimate the slope **by eye**, the way you would off a printed plot: rise over run between the first and last stacked means. Then fit it with `np.polyfit`: it finds the line that minimizes the summed squared vertical misses across **all 25 trials**, not just the two endpoints. (Where that recipe comes from is the Week 2 meeting; for now we use it the way we use a calculator.)

In [ ]:
# By eye: rise over run between the endpoint means
rise = stats_T2['mean'].loc[1.2] - stats_T2['mean'].loc[0.4]  # s^2
run = 1.2 - 0.4                                               # m
slope_eye = rise / run
g_eye = 4 * np.pi**2 / slope_eye

print(f'by-eye slope:  {slope_eye:.3f} s^2/m')
print(f'g by eye:      {g_eye:.3f} m/s^2')

`np.polyfit(x, y, 1)` fits a degree-1 polynomial (a straight line) to the points and returns its coefficients, slope first.

In [ ]:
# Least squares on all 25 trials
slope_fit, intercept_fit = np.polyfit(df['length_m'], df['period_sq'], 1)
g_fit = 4 * np.pi**2 / slope_fit
epsilon_fit = intercept_fit / slope_fit  # the length offset, in metres

print(f'fitted slope:      {slope_fit:.4f} s^2/m')
print(f'fitted intercept:  {intercept_fit:.4f} s^2')
print(f'length offset:     {epsilon_fit * 1000:.1f} mm   (intercept / slope)')
print(f'g from the fit:    {g_fit:.3f} m/s^2')
print(f'g by eye, again:   {g_eye:.3f} m/s^2')

In [ ]:
# All 25 trials plus the fitted line
fig = px.scatter(
    df,
    x='length_m',
    y='period_sq',
    title='T squared vs l: 25 trials and the least-squares line',
    labels={'length_m': 'Pendulum length l (m)', 'period_sq': 'T squared (s^2)'},
)
fig.update_traces(marker=dict(size=10))
fig.data[0].name = 'measured trials'
fig.data[0].showlegend = True

L_line = np.linspace(0.3, 1.3, 50)  # 50 evenly spaced lengths for drawing the line
fig.add_scatter(
    x=L_line,
    y=slope_fit * L_line + intercept_fit,
    mode='lines',
    line=dict(dash='dash'),
    name='least-squares fit',
)
fig.show()

**Figure description:** The 25 measured $T^2$ values (circular markers) with the least-squares line (dashed) overlaid. On the sample CSV you should find the line passes through the middle of every five-trial cluster: slope 4.086 s$^2$/m, intercept $-0.033$ s$^2$, as printed by the fitting cell above. On the in-class CSV, describe your own fitted slope and intercept the same way. The takeaway is that one straight line describes all 25 trials, with misses no larger than the trial-to-trial scatter.

> **Self-check (Part 4).** `slope_eye` should land between 3.9 and 4.2 s$^2$/m, and `slope_fit` between 4.05 and 4.12 s$^2$/m. Both $g$ values should land between 9.5 and 9.8 m/s$^2$ (ours: 9.706 by eye, 9.661 fitted). If a $g$ came out near 0.10, you computed slope$/4\pi^2$: flip the division. If `slope_fit` came out near 0.24, the `x` and `y` arguments to `polyfit` are swapped.

**Question 4.1.** How close did the by-eye slope come to the least-squares slope, in percent? Give one reason the least-squares answer is still preferable even when the two agree well.

*(Your answer):*

**Question 4.2.** On the sample CSV you should find the fit returned an intercept of about $-0.03$ s$^2$, which works out to a length offset of about $-8$ mm; Part 5 will show you that the fit pins the intercept only to about $\pm 0.03$ s$^2$, so this offset is about one error bar from zero. On the in-class CSV, read off your own fitted intercept and its length offset the same way, and compare your recovered $g$ against 9.81 m/s$^2$. Say in one sentence what your intercept means about this apparatus. Then suppose a different dataset returned an intercept of $+0.5$ s$^2$, far outside any error bar. Name one systematic problem with that setup or procedure that could do it. (Hint: think about what "length" gets measured with a fat bob on a string, or what a stopwatch habit could do to every trial equally.)

*(Your answer):*

## Part 5: How wrong could we be? (~20 min)

A measured number needs an uncertainty. The fit itself can tell us how well the data pin down its two coefficients: calling `np.polyfit(..., cov=True)` also returns a **covariance matrix**. The square root of its top-left entry is the standard error of the slope, $\sigma_{\text{slope}}$; the square root of its bottom-right entry is the standard error of the intercept, $\sigma_{\text{intercept}}$.

Because $g = 4\pi^2/\text{slope}$, a small relative error in the slope produces the same relative error in $g$:

$$\sigma_g = g \cdot \frac{\sigma_{\text{slope}}}{\text{slope}}$$

The cell below computes both, then asks: is the accepted value 9.81 m/s$^2$ inside our error bar, and if not, by how many error bars does it miss?

In [ ]:
# The same fit, but also return the covariance matrix
coeffs, cov = np.polyfit(df['length_m'], df['period_sq'], 1, cov=True)
slope_fit, intercept_fit = coeffs

slope_sigma = np.sqrt(cov[0, 0])      # standard error of the slope
intercept_sigma = np.sqrt(cov[1, 1])  # standard error of the intercept
g_fit = 4 * np.pi**2 / slope_fit
g_sigma = g_fit * slope_sigma / slope_fit

gap_in_sigmas = (9.81 - g_fit) / g_sigma

print(f'slope:  {slope_fit:.4f} +/- {slope_sigma:.4f} s^2/m')
print(f'g:      {g_fit:.3f} +/- {g_sigma:.3f} m/s^2')
print(f'accepted g = 9.810 m/s^2 sits {gap_in_sigmas:.2f} error bars above our value')
print(f'intercept: {intercept_fit:.4f} +/- {intercept_sigma:.4f} s^2 ({abs(intercept_fit / intercept_sigma):.1f} error bars from zero)')

**Reading the result honestly.** On the sample CSV this gives $g = 9.66 \pm 0.09$ m/s$^2$, with 9.81 sitting about 1.75 error bars away. That is not a failure, and nothing in the analysis is "wrong": for an honest experiment, landing 1.75 error bars from the truth happens roughly one run in twelve, purely by chance. The error bar exists precisely to tell you how far a single honest run can land. The forbidden move is the opposite one: nudging data, dropping trials, or shopping for an analysis until the answer hits 9.81. The result of a run is whatever the data say, plus an honest error bar.

> **Self-check (Part 5).** `slope_sigma` should land between 0.03 and 0.05 s$^2$/m, and `g_sigma` between 0.07 and 0.10 m/s$^2$. With the sample CSV, $g = 9.661 \pm 0.085$ m/s$^2$, and the gap to 9.81 is 1.7 to 1.8 error bars. If `g_sigma` came out near 0.003, you forgot the square root on `cov[0, 0]`. The intercept should print as $-0.0332 \pm 0.0305$ s$^2$, about 1.1 error bars from zero: on this dataset the length offset is not distinguishable from no offset at all.

**Question 5.1.** Report your final result in standard form, $g = \text{value} \pm \text{uncertainty}$ m/s$^2$, with units and a sensible number of digits (give the uncertainty one or two significant figures and stop the value at the same decimal place). Does 9.81 fall within one error bar of your value? Within two?

*(Your answer):*

**Question 5.2.** Two stories fit a low result: (a) chance, this run simply landed low; (b) something systematic, for example every period read slightly long (a stopwatch habit) or every length measured slightly short (measuring to the knot instead of the bob's center). The error bar from the fit only knows about story (a). Describe one concrete follow-up measurement that could tell the two stories apart. (One good answer uses data you may already have: the Wk 2 in-class CSV is an independent run on the same apparatus.)

*(Your answer):*



## Part 6: Reflection (~15 min)

Week 1 opened with $g = GM/R^2$ on the board, and this table of its hidden assumptions:

| Hidden assumption | Real departure | Reduction step |
|---|---|---|
| Earth is a point mass / spherically symmetric | Earth is an ellipsoid | Latitude correction |
| Observer is at the reference surface | Observer is above sea level | Free-air |
| Nothing between observer and reference | Rock between observer and sea level | Bouguer |
| Surrounding terrain is flat | Hills and valleys exist | Terrain |
| Lithosphere is rigid on a rigid mantle | Lithosphere floats | Isostatic |
| Instrument is stable in time | Springs creep, temperature drifts | Drift |
| Earth-Moon-Sun geometry is fixed | It is not | Tidal |

**Question 6.1.** Go down the table row by row: which rows did HW0 engage or test? (Think about which *side* of $g = GM/R^2$ the pendulum measures, and notice that nothing in this notebook ever computed $GM/R^2$.)

*(Your answer):*

**Question 6.2.** The pendulum adds one assumption not in the table, and the data-collection protocol guards it by keeping every swing under 15 degrees. What is the assumption? What do you expect happens to the measured $T$ (and therefore to your $g$) if someone launches the pendulum from 60 degrees? (The Wk 2 series-expansion warm-up takes this exact assumption apart: $\sin\theta \approx \theta$.)

*(Your answer):*

**Question 6.3.** *(Optional, ungraded.)* What surprised you most: the Python, the pendulum, or the error bar?

*(Your answer):*

## How to submit (~5 min)

1. Runtime > Run all. Every cell must run top to bottom with no errors.
2. Check that every *(Your answer):* is filled in and every self-check passed.
3. File > Download > Download .ipynb.
4. Rename the file `HW0_LASTNAME.ipynb` (for example `HW0_Smith.ipynb`).
5. Upload to the Brightspace HW0 dropbox by **Wednesday, September 16, 2026, 11:59 PM**.

Stuck, or something looks broken? Climb the getting-unstuck ladder at the top of this notebook. Rung 4 is the **Ask the Class (General Q&A)** discussion topic; a classmate may have hit the same snag.